CONSTANTS

In [ ]:
LOCAL_DIR = "data/hotel_bookings_with_holidays.csv"
KAGGLE_DIR = "/kaggle/input/datasets/omarhashem80/hotel-booking-demand/hotel_bookings_with_holidays.csv"

## 🔴Import Libraries 📦

In [ ]:
import numpy as np
from scipy.stats import chi2_contingency, f_oneway, kruskal
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import  DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold, cross_validate
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import clone
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
import joblib
import warnings
warnings.filterwarnings('ignore')

## 🔴Load Dataset 🗂️

In [ ]:
df = pd.read_csv(LOCAL_DIR)

## 🔴Features Understanding 📖

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
for col in df.select_dtypes(include='object').columns:
  print(df[col].value_counts(),"*"*50,sep='\n')

## 🔴Cleaning🕵️

In [ ]:
df.isna().mean() * 100

In [ ]:
df["children"] = df["children"].fillna(0)

# country → unknown category
df["country"] = df["country"].fillna("Unknown")

# holiday-related
df["is_holiday"] = df["is_holiday"].fillna(0)
df["days_to_next_holiday"] = df["days_to_next_holiday"].fillna(-1)
df["days_from_last_holiday"] = df["days_from_last_holiday"].fillna(-1)

In [ ]:
df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"])
df["arrival_date"] = pd.to_datetime(df["arrival_date"])

In [ ]:
# company is mostly missing → often useless
df = df.drop(columns=["company"])
df = df.drop(columns=["reservation_status"])

In [ ]:
# integers
int_cols = [
    "is_canceled", "lead_time", "arrival_date_year",
    "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies",
    "is_repeated_guest", "previous_cancellations",
    "previous_bookings_not_canceled", "booking_changes",
    "days_in_waiting_list", "required_car_parking_spaces",
    "total_of_special_requests"
]

for col in int_cols:
    df[col] = pd.to_numeric(df[col], downcast="integer")

# floats
float_cols = ["adr"]
for col in float_cols:
    df[col] = pd.to_numeric(df[col], downcast="float")


# boolean-like
df["is_holiday"] = df["is_holiday"].astype("int8")

In [ ]:
df["total_guests"] = df["adults"] + df["children"] + df["babies"]
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]

In [ ]:
# CATEGORICAL COLUMNS
cat_cols = [
    "hotel", "arrival_date_month", "meal", "country",
    "market_segment", "distribution_channel",
    "reserved_room_type", "assigned_room_type",
    "deposit_type", "customer_type", "is_canceled"
]

for col in cat_cols:
    df[col] = df[col].astype("category")

In [ ]:
df[df["total_guests"] == 0]

In [ ]:
df["agent"].fillna(0, inplace=True)

In [ ]:
# IDs as integers (not float)
df["agent"] = df["agent"].astype("int32")

In [ ]:
df = df[df["total_guests"] > 0]

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
df.head()

In [ ]:
df.info()

## 🔴EDA

In [ ]:
nums = df.select_dtypes(include=np.number).columns.tolist()
cats = df.select_dtypes(include='category').columns.tolist()
print("Numerical Columns:", nums)
print("-" * 500)
print("Categorical Columns:", cats)

### 🔴Univariante Analysis **👆**

In [ ]:
def create_histogram_with_boxplot(df, x):
    """
    Create a Plotly figure containing a histogram and a box plot for a specified column in a DataFrame.
    Parameters:
    - df (pd.DataFrame): The DataFrame containing the data.
    - x (str): The name of the column to plot.

    Returns:
    - fig (go.Figure): Plotly figure containing the histogram and box plot.
    """
    fig = make_subplots(rows=2, cols=1)
    fig.add_trace(go.Histogram(x=df[x], name='Histogram'), row=1, col=1)
    fig.add_trace(go.Box(x=df[x], name="Box"), row=2, col=1)
    fig.update_layout(
        title=f'Distribution of {x}',
        height=600,
    )
    return fig


def create_pie_chart(df, cat):
    """
    Create a Plotly pie chart for the distribution of categories in a specified column of a DataFrame.

    Parameters:
    - df (pd.DataFrame): The DataFrame containing the data.
    - cat (str): The name of the categorical column to plot.

    Returns:
    - fig (px.pie): Plotly pie chart.
    """
    fig = px.pie(names=df[cat], hole=0.3, title=f"Distribution of {cat}")
    fig.update_traces(textinfo='percent+label')
    fig.update_layout(height=800)  # Adjust height and width as desired
    return fig

#### 🟢Numerical🔢

In [ ]:
create_histogram_with_boxplot(df, nums[0])

In [ ]:
create_histogram_with_boxplot(df, nums[1])

In [ ]:
create_histogram_with_boxplot(df, nums[2])

In [ ]:
create_histogram_with_boxplot(df, nums[3])

In [ ]:
create_histogram_with_boxplot(df, nums[4])

In [ ]:
create_histogram_with_boxplot(df, nums[5])

In [ ]:
create_histogram_with_boxplot(df, nums[6])

In [ ]:
create_histogram_with_boxplot(df, nums[7])

In [ ]:
create_histogram_with_boxplot(df, nums[8])

In [ ]:
create_histogram_with_boxplot(df, nums[9])

In [ ]:
create_histogram_with_boxplot(df, nums[10])

In [ ]:
create_histogram_with_boxplot(df, nums[11])

In [ ]:
create_histogram_with_boxplot(df, nums[12])

In [ ]:
create_histogram_with_boxplot(df, nums[13])

In [ ]:
create_histogram_with_boxplot(df, nums[14])

In [ ]:
create_histogram_with_boxplot(df, nums[15])

In [ ]:
create_histogram_with_boxplot(df, nums[16])

In [ ]:
create_histogram_with_boxplot(df, nums[17])

In [ ]:
create_histogram_with_boxplot(df, nums[18])

In [ ]:
create_histogram_with_boxplot(df, nums[19])

In [ ]:
create_histogram_with_boxplot(df, nums[20])

In [ ]:
create_histogram_with_boxplot(df, nums[21])

In [ ]:
create_histogram_with_boxplot(df, nums[22])

#### 🔵Categorical

In [ ]:
create_pie_chart(df, cats[0])

In [ ]:
create_pie_chart(df, cats[1])

In [ ]:
create_pie_chart(df, cats[2])

In [ ]:
create_pie_chart(df, cats[3])

In [ ]:
# replace the plot
create_pie_chart(df, cats[4])

In [ ]:
create_pie_chart(df, cats[5])

In [ ]:
create_pie_chart(df, cats[6])

In [ ]:
create_pie_chart(df, cats[7])

In [ ]:
create_pie_chart(df, cats[8])

In [ ]:
create_pie_chart(df, cats[9])

In [ ]:
create_pie_chart(df, cats[10])

### 🔴Bivariante Analysis✌️

#### 🟢Numerical vs Numerical

In [ ]:
px.imshow(df[nums].corr(numeric_only=True),text_auto='.2f',width=1000, height=800)

In [ ]:
# fig = px.scatter(df, x='departure delay in minutes', y='arrival delay in minutes', trendline='ols', title='Relationship between departure delay and arrival delay in minutes')
# fig.update_traces(marker=dict(size=8, opacity=0.5))
# fig.show()

#### ⚫Categorical vs Numerical


In [ ]:
nums

In [ ]:
cats

In [ ]:
dff = df.groupby('satisfaction')[['age']].mean().reset_index()
dff

In [ ]:
fig = px.bar(dff, x='satisfaction', y='age', title='Mean Age by Satisfaction',
             labels={'satisfaction': 'Satisfaction', 'age': 'Mean Age'}, text_auto='.2f' )
fig.update_layout(xaxis={'categoryorder':'total ascending'})
fig.show()

#### ⚫Categorical vs Categorical

In [ ]:
satisfaction_by_gender = df.groupby('gender')['satisfaction'].value_counts(normalize=True).unstack().reset_index()
satisfaction_by_gender

In [ ]:
satisfaction_by_gender_melted = satisfaction_by_gender.melt(id_vars='gender', var_name='satisfaction', value_name='proportion')
satisfaction_by_gender_melted

In [ ]:
fig = px.bar(satisfaction_by_gender_melted, x='gender', y='proportion', color='satisfaction', barmode='group',text_auto='.2f' ,
             title='Distribution of Satisfaction by Gender', labels={'gender': 'Gender', 'proportion': 'Proportion', 'satisfaction': 'Satisfaction'})
fig.show()

In [ ]:
cats

In [ ]:
satisfaction_by_customer_type = df.groupby('customer type')['satisfaction'].value_counts(normalize=True).unstack().reset_index()
satisfaction_by_customer_type

In [ ]:
satisfaction_by_customer_type_melted = satisfaction_by_customer_type.melt(id_vars='customer type', var_name='satisfaction', value_name='proportion')
satisfaction_by_customer_type_melted

In [ ]:
fig = px.bar(satisfaction_by_customer_type_melted, x='customer type', y='proportion', color='satisfaction', barmode='group',text_auto='.2f' ,
                title='Distribution of Satisfaction by Customer Type', labels={'customer type': 'Customer Type', 'proportion': 'Proportion', 'satisfaction': 'Satisfaction'})
fig.show()

In [ ]:
satisfaction_by_class = df.groupby('class')['satisfaction'].value_counts(normalize=True).unstack().reset_index()
satisfaction_by_class

In [ ]:
satisfaction_by_class_melted = satisfaction_by_class.melt(id_vars='class', var_name='satisfaction', value_name='proportion')
satisfaction_by_class_melted

In [ ]:
fig = px.bar(satisfaction_by_class_melted, x='class', y='proportion', color='satisfaction', barmode='group',text_auto='.2f' ,
             title='Distribution of Satisfaction by Class', labels={'class': 'Class', 'proportion': 'Proportion', 'satisfaction': 'Satisfaction'})
fig.show()

In [ ]:
satisfaction_by_type_of_travel = df.groupby('type of travel')['satisfaction'].value_counts(normalize=True).unstack().reset_index()
satisfaction_by_type_of_travel

In [ ]:
satisfaction_by_type_of_travel_melted = satisfaction_by_type_of_travel.melt(id_vars='type of travel', var_name='satisfaction', value_name='proportion')
satisfaction_by_type_of_travel_melted

In [ ]:
fig = px.bar(satisfaction_by_type_of_travel_melted, x='type of travel', y='proportion', color='satisfaction', barmode='group',text_auto='.2f' ,
             title='Distribution of Satisfaction by Type of Travel', labels={'type of travel': 'Type of Travel', 'proportion': 'Proportion', 'satisfaction': 'Satisfaction'})
fig.show()

In [ ]:
cats.remove('is_canceled')

In [ ]:
for cat in cats:
    # Create a contingency table
    contingency_table = pd.crosstab(df[cat], df['is_canceled'])

    # Perform chi-square test
    chi2, p, dof, expected = chi2_contingency(contingency_table)

    # Print results
    print(f"Feature: {cat}")
    print(f"Chi-square statistic: {chi2:.4f}")
    print(f"P-value: {p:.6f}")

    # Significance level
    alpha = 0.05

    # Interpretation
    if p < alpha:
        print(f"There is a significant association between \033[94m{cat}\033[0m and \033[94mis_canceled\033[0m.")
    else:
        print(f"There is \033[96mno\033[0m significant association between \033[94m{cat}\033[0m and \033[94mis_canceled\033[0m.")

    print('*' * 120)

## 🔴Preprocessing

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

### 🔴Encoding

In [ ]:
# Target
target_col = "is_canceled"

# Binary (0/1 or yes/no style)
binary_cols = [
    "is_repeated_guest",
    "is_holiday"
]

# Nominal (no order)
nominal_cols = [
    "hotel",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "assigned_room_type",
    "deposit_type",
    "customer_type"
]

# Ordinal (has meaning/order)
ordinal_cols = [
    "arrival_date_month"  # will encode month order
]

# Numeric
numeric_cols = [
    "lead_time",
    "arrival_date_year",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "agent",
    "days_in_waiting_list",
    "adr",
    "required_car_parking_spaces",
    "total_of_special_requests",
    "days_to_next_holiday",
    "days_from_last_holiday",
    "total_guests",
    "total_nights"
]

In [ ]:
binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder())
])

nominal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[month_order]))
])

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Full preprocessor
preprocessor = ColumnTransformer([
    ('binary', binary_pipeline, binary_cols),
    ('nominal', nominal_pipeline, nominal_cols),
    ('ordinal', ordinal_pipeline, ordinal_cols),
    ('numeric', numeric_pipeline, numeric_cols)
])

In [ ]:
X = df.drop(target_col,axis=1)
le = LabelEncoder()
y = le.fit_transform(df[target_col])

## 🔴Modeling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
def extract_cv_metrics(grid, param_name):
    results = grid.cv_results_

    # parameter values
    param_values = results[f'param_{param_name}'].data

    # accuracy
    train_acc = results['mean_train_score']
    val_acc = results['mean_test_score']

    # log loss (may not exist depending on scoring)
    train_loss = None
    val_loss = None

    if 'mean_train_neg_log_loss' in results:
        train_loss = -results['mean_train_neg_log_loss']
        val_loss = -results['mean_test_neg_log_loss']

    return param_values, train_acc, val_acc, train_loss, val_loss

In [ ]:
def plot_accuracy(name, param_name, param_values, train_acc, val_acc, save_path=None):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=param_values, y=train_acc,
        mode='lines+markers',
        name='Train Accuracy'
    ))

    fig.add_trace(go.Scatter(
        x=param_values, y=val_acc,
        mode='lines+markers',
        name='Validation Accuracy'
    ))

    fig.update_layout(
        title=f"{name} - Accuracy vs {param_name}",
        xaxis_title=param_name,
        yaxis_title="Accuracy",
        template="plotly_white"
    )

    if save_path:
        fig.write_html(f"{save_path}_accuracy.html")
        print(f"Saved accuracy plot to {save_path}_accuracy.html")

    fig.show()


def plot_loss(name, param_name, param_values, train_loss, val_loss, save_path=None):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=param_values, y=train_loss,
        mode='lines+markers',
        name='Train Log Loss'
    ))

    fig.add_trace(go.Scatter(
        x=param_values, y=val_loss,
        mode='lines+markers',
        name='Validation Log Loss'
    ))

    fig.update_layout(
        title=f"{name} - Log Loss vs {param_name}",
        xaxis_title=param_name,
        yaxis_title="Log Loss",
        template="plotly_white"
    )

    if save_path:
        fig.write_html(f"{save_path}_loss.html")
        print(f"Saved loss plot to {save_path}_loss.html")

    fig.show()

In [ ]:
def run_model_pipeline(name, model, param_grid, param_name, save_path=None):
    print(f"\n{'='*30} {name} {'='*30}")

    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    grid = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='accuracy',
        return_train_score=True, 
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    print(f"Best CV Accuracy: {grid.best_score_:.4f}")
    print(f"Best Params: {grid.best_params_}")

    best_pipeline = grid.best_estimator_

    param_values, train_acc, val_acc, train_loss, val_loss = extract_cv_metrics(
        grid, param_name
    )

    # plots
    plot_accuracy(name, param_name, param_values, train_acc, val_acc, save_path)

    if train_loss is not None:
        plot_loss(name, param_name, param_values, train_loss, val_loss, save_path)

    # fit best model
    best_pipeline.fit(X_train, y_train)

    y_train_pred = best_pipeline.predict(X_train)
    y_test_pred = best_pipeline.predict(X_test)

    print("\nTrain Report:")
    print(classification_report(y_train, y_train_pred))

    print("Test Report:")
    print(classification_report(y_test, y_test_pred))

    return grid.best_score_, best_pipeline

In [ ]:
results = {}

# Logistic Regression
score, model_lr = run_model_pipeline(
    "Logistic Regression",
    LogisticRegression(max_iter=1000),
    {'model__C': [0.1, 1, 10]},
    'model__C',
    save_path="rf_plots"
)
results['LR'] = (score, model_lr)

In [ ]:
# # Random Forest
# score, model_rf = run_model_pipeline(
#     "Random Forest",
#     RandomForestClassifier(random_state=42, n_jobs=-1),
#     {
#         'model__n_estimators': [100, 200],
#         'model__max_depth': [None, 10]
#     },
#     'model__n_estimators',
#     save_path="rf_plots"
# )
# results['RF'] = (score, model_rf)

In [ ]:
# XGBoost
score, model_xgb = run_model_pipeline(
    "XGBoost",
    XGBClassifier(eval_metric='mlogloss', random_state=42),
    {
        'model__n_estimators': [100, 200],
        'model__max_depth': [3, 5]
    },
    'model__n_estimators',
    save_path="rf_plots"
)
results['XGB'] = (score, model_xgb)

In [ ]:
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

In [ ]:
# # CatBoost
# score, model_cb = run_model_pipeline(
#     "CatBoost",
#     CatBoostClassifier(verbose=0, random_state=42),
#     {
#         'model__iterations': [200, 500],
#         'model__depth': [6, 8]
#     },
#     'model__iterations',
#     save_path="rf_plots"
# )
# results['CB'] = (score, model_cb)
cat = CatBoostClassifier(iterations=500)
cat.fit(X_train, y_train, cat_features=cat_cols)

y_pred_cat = cat.predict(X_test)

acc_cat = accuracy_score(y_test, y_pred_cat)
conf = confusion_matrix(y_test, y_pred_cat)
clf_report = classification_report(y_test, y_pred_cat)

In [ ]:
print(clf_report)

In [ ]:
best_name = max(results, key=lambda x: results[x][0])
best_model = results[best_name][1]

print(f"\n🏆 FINAL BEST MODEL: {best_name}")

In [ ]:
joblib.dump(best_model, "best_model.pkl")
joblib.dump(le, "label_encoder.pkl")

## 🔴Report